In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Data Analysis**

In [2]:
import os
import random
from transformers import MarianMTModel, MarianTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

# 1. Load the English and Dutch data from the files (limited to 10,000)
def load_data(en_path, nl_path, num_samples=10000):
    with open(en_path, 'r', encoding='utf-8') as file_en, open(nl_path, 'r', encoding='utf-8') as file_nl:
        en_sentences = [line.strip() for line in file_en.readlines()]
        nl_sentences = [line.strip() for line in file_nl.readlines()]

    # Ensure both datasets have the same number of sentences
    assert len(en_sentences) == len(nl_sentences), "Source and target files have different lengths"

    # Randomly sample 10,000 sentence pairs
    sampled_indices = random.sample(range(len(en_sentences)), num_samples)

    en_sentences = [en_sentences[i] for i in sampled_indices]
    nl_sentences = [nl_sentences[i] for i in sampled_indices]

    return en_sentences, nl_sentences

In [3]:

# 2. Define a Dataset Class
class TranslationDataset(Dataset):
    def __init__(self, tokenizer, src_texts, tgt_texts, max_len=128):
        self.tokenizer = tokenizer
        self.src_texts = src_texts  # English sentences
        self.tgt_texts = tgt_texts  # Dutch sentences
        self.max_len = max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_text = self.src_texts[idx]
        tgt_text = self.tgt_texts[idx]

        # Tokenize the source (English) and target (Dutch) texts
        src_enc = self.tokenizer(src_text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        tgt_enc = self.tokenizer(tgt_text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")

        # Labels need to be the tokenized target sentences with -100 where padding tokens are
        labels = tgt_enc.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": src_enc.input_ids.squeeze(),
            "attention_mask": src_enc.attention_mask.squeeze(),
            "labels": labels,
        }

In [4]:

# 3. Load the MarianMT model and tokenizer
model_name = "Helsinki-NLP/opus-mt-en-nl"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/316M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:

# 4. Load your dataset (limit to 10,000 sentence pairs)
en_sentences, nl_sentences = load_data(r"drive/My Drive/Project/Project Data/CCMatrix.en-nl.en",
                                       r"drive/My Drive/Project/Project Data/CCMatrix.en-nl.nl",
                                       num_samples=5000)

In [ ]:

# Split into train and validation sets (90% train, 10% validation)
split_size = int(0.9 * len(en_sentences))
train_en, val_en = en_sentences[:split_size], en_sentences[split_size:]
train_nl, val_nl = nl_sentences[:split_size], nl_sentences[split_size:]

# Create datasets
train_dataset = TranslationDataset(tokenizer, train_en, train_nl)
val_dataset = TranslationDataset(tokenizer, val_en, val_nl)


In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
!pip install datasets --upgrade

In [ ]:
import datasets
print(datasets.__version__)

3.1.0


In [ ]:
!pip uninstall datasets -y
!pip install datasets==3.1.0 --force-reinstall --no-cache-dir

Found existing installation: datasets 3.1.0
Uninstalling datasets-3.1.0:
  Successfully uninstalled datasets-3.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 23.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 159.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 210.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 276.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 287.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 319.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 282.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 114.2 MB/s eta 0:

In [ ]:
from datasets import load_metric
# Load BLEU metric
bleu_metric = load_metric("bleu")

# Define a function for computing metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as it may not be correctly handled by tokenizer
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BLEU expects a list of references; decoded_labels should be nested
    decoded_labels = [[label] for label in decoded_labels]

    # Calculate BLEU score
    bleu_score = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["bleu"]}


ImportError: cannot import name 'load_metric' from 'datasets' (/usr/local/lib/python3.10/dist-packages/datasets/__init__.py)

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

# 5. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./marian_en_nl_10000",   # Save directory
    evaluation_strategy="epoch",         # Evaluate every epoch
    learning_rate=5e-5,                  # Learning rate
    per_device_train_batch_size=8,       # Batch size for training
    per_device_eval_batch_size=8,        # Batch size for evaluation
    num_train_epochs=10,                  # Number of epochs
    weight_decay=0.01,                   # Weight decay for regularization
    save_total_limit=3,                  # Limit the number of saved checkpoints         # Predict with text generation
    logging_dir="./logs",                # Directory for storing logs
    logging_steps=100,                   # Log every 100 steps
    fp16=True                            # Enable mixed precision training
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


NameError: name 'compute_metrics' is not defined

In [ ]:

# 7. Train the model
trainer.train()


  0%|          | 0/1689 [00:00<?, ?it/s]

{'loss': 3.8748, 'grad_norm': 8.179119110107422, 'learning_rate': 4.70396684428656e-05, 'epoch': 0.18}
{'loss': 3.0917, 'grad_norm': 9.231163024902344, 'learning_rate': 4.40793368857312e-05, 'epoch': 0.36}
{'loss': 2.7529, 'grad_norm': 7.8218159675598145, 'learning_rate': 4.111900532859681e-05, 'epoch': 0.53}
{'loss': 2.5789, 'grad_norm': 11.20527172088623, 'learning_rate': 3.8158673771462406e-05, 'epoch': 0.71}


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[67027]], 'forced_eos_token_id': 0}


{'loss': 2.478, 'grad_norm': 8.861053466796875, 'learning_rate': 3.5198342214328006e-05, 'epoch': 0.89}


  0%|          | 0/63 [00:00<?, ?it/s]

...

{'loss': 1.5336, 'grad_norm': 6.598458290100098, 'learning_rate': 5.595026642984015e-06, 'epoch': 2.66}
{'loss': 1.5059, 'grad_norm': 7.666136264801025, 'learning_rate': 2.634695085849615e-06, 'epoch': 2.84}


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[67027]], 'forced_eos_token_id': 0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.9359321594238281, 'eval_runtime': 100.0337, 'eval_samples_per_second': 4.998, 'eval_steps_per_second': 0.63, 'epoch': 3.0}
{'train_runtime': 7552.6212, 'train_samples_per_second': 1.787, 'train_steps_per_second': 0.224, 'train_loss': 2.1096829854221015, 'epoch': 3.0}


TrainOutput(global_step=1689, training_loss=2.1096829854221015, metrics={'train_runtime': 7552.6212, 'train_samples_per_second': 1.787, 'train_steps_per_second': 0.224, 'total_flos': 457627926528000.0, 'train_loss': 2.1096829854221015, 'epoch': 3.0})

In [ ]:

# 8. Evaluate the model
trainer.evaluate()


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.9359321594238281,
 'eval_runtime': 88.3241,
 'eval_samples_per_second': 5.661,
 'eval_steps_per_second': 0.713,
 'epoch': 3.0}

In [ ]:
# Optionally, you can also save the model
model.save_pretrained("./marian_en_nl_finetuned_10000_2")
tokenizer.save_pretrained("./marian_en_nl_finetuned_10000_2")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[67027]], 'forced_eos_token_id': 0}


('./marian_en_nl_finetuned_10000\\tokenizer_config.json',
 './marian_en_nl_finetuned_10000\\special_tokens_map.json',
 './marian_en_nl_finetuned_10000\\vocab.json',
 './marian_en_nl_finetuned_10000\\source.spm',
 './marian_en_nl_finetuned_10000\\target.spm',
 './marian_en_nl_finetuned_10000\\added_tokens.json')

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# 1. Load the fine-tuned MarianMT model and tokenizer
model_name = "./marian_en_nl_finetuned_10000"  # Directory where your fine-tuned model is saved
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# 2. Function to translate English to Dutch
def translate(text, model, tokenizer, max_length=128):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Generate translation
    translated_tokens = model.generate(**inputs)

    # Decode the tokens back to a string
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_text

# 3. Input sentence (English)
input_sentence = "Hi, I’m Winston, a cuddly rabbit that was once the proud assistant of a forgetful magician named Merlin."

# 4. Translate the sentence
output_translation = translate(input_sentence, model, tokenizer)

# 5. Print the result
print("Input Sentence:", input_sentence)
print("Translated Sentence:", output_translation)


Input Sentence: Hi, I’m Winston, a cuddly rabbit that was once the proud assistant of a forgetful magician named Merlin.
Translated Sentence: Hoi, ik ben Winston, een knullig konijn dat eens de trotse assistant was van een vergeten tovenaant die Merlin noemde.


In [ ]:
#English test as input and Dutch as output, check if its a string, not a number, not special characters, length?
#Output within 5 seconds


from langdetect import detect, LangDetectException
import unittest
import time
from transformers import MarianMTModel, MarianTokenizer

# Sample data for testing


class TestMachineTranslation(unittest.TestCase):
    def setUp(self):
        self.tokenizer = MarianTokenizer.from_pretrained("./marian_en_nl_finetuned_10000")
        self.model = MarianMTModel.from_pretrained("./marian_en_nl_finetuned_10000")

    def test_translation_time(self):
        sentences: [
                "This is a sentence to translate from English to Dutch.",
                "Hi, I’m Winston, a cuddly rabbit that was once the proud assistant of a forgetful magician named Merlin. One day, Merlin put me in his hat and forgot. I found myself in a land of mismatched socks, lost keys, and odd handkerchiefs. With some effort (and nibbling on some old capes) , I finally made my way out and hopped over to the shelter, hoping for a quieter life away from magical blunders. Now, I’m looking for a home where the only tricks are those involving carrot treats.",
                "It was easy to spot her. All you needed to do was look at her socks. They were never a matching pair. One would be green while the other would be blue. One would reach her knee while the other barely touched her ankle. Every other part of her was perfect, but never the socks. They were her micro act of rebellion. You can decide what you want to do in life, but I suggest doing something that creates. Something that leaves a tangible thing once you're done. That way even after you're gone, you will still live on in the things you created. According to the caption on the bronze marker placed by the Multnomah Chapter of the Daughters of the American Revolution on May 12, 1939, “College Hall (is) the oldest building in continuous use for Educational purposes west of the Rocky Mountains. Here were educated men and women who have won recognition throughout the world in all the learned professions.” The leather jacked showed the scars of being his favorite for years. It wore those scars with pride, feeling that they enhanced his presence rather than diminishing it. The scars gave it character and had not overwhelmed to the point that it had become ratty. The jacket was in its prime and it knew it. It was the best compliment that he'd ever received although the person who gave it likely never knew. It had been an off-hand observation on his ability to hold a conversation and actually add pertinent information to it on practically any topic. Although he hadn't consciously strived to be able to do so, he'd started to voraciously read the news when he couldn't keep up on topics his friends discussed because their conversations went above his head. The fact that someone had noticed enough to compliment him that he could talk intelligently about many topics meant that he had succeeded in his quest to be better informed. There wasn't a whole lot more that could be done. It had become a wait-and-see situation with the final results no longer in her control. That didn't stop her from trying to control the situation. She demanded that things be done as she desperately tried to control what couldn't be. Don't forget that gifts often come with costs that go beyond their purchase price. When you purchase a child the latest smartphone, you're also committing to a monthly phone bill. When you purchase the latest gaming system, you're likely not going to be satisfied with the games that come with it for long and want to purchase new titles to play. When you buy gifts it's important to remember that some come with additional costs down the road that can be much more expensive than the initial gift itself. He took a sip of the drink. He wasn't sure whether he liked it or not, but at this moment it didn't matter. She had made it especially for him so he would have forced it down even if he had absolutely hated it. That's simply the way things worked. She made him a new-fangled drink each day and he took a sip of it and smiled, saying it was excellent. There was nothing to indicate Nancy was going to change the world. She looked like an average girl going to an average high school. It was the fact that everything about her seemed average that would end up becoming her superpower. I'm so confused by your ridiculous meltdown that I must insist on some sort of explanation for your behavior towards me. It just doesn't make any sense. There's no way that I deserved the treatment you gave me without an explanation or an apology for how out of line you have been. I'm heading back to Colorado tomorrow after being down in Santa Barbara over the weekend for the festival there.  ",
                ]
        for input_text in sentences:
            with self.subTest(input_text=input_text):
                print(f"Sentence input: {input_text}")
                start = time.time()
                translated_sentence = translate(input_text, model, tokenizer)
                elapsed_time = time.time() - start
                print(f"Sentence output: {translated_sentence}, time elapsed: {elapsed_time}")
                self.assertLess(elapsed_time, 5, "Translation took more than 5 seconds")

    def test_input_is_english(self):
        #sentence = "This is a test sentence to check the input language."
        sentence = "Hoi, hoe hoe hoe hoe het?"

        try:
            # Detect the language
            detected_language = detect(sentence)

            # Assert that it’s English
            self.assertEqual(detected_language, 'en', "Input is not English")

        except LangDetectException:
            # If detection fails, handle the error (e.g., log it or fail the test)
            self.fail("Language detection failed")

    def test_model_prediction(self):
        test_en_sentences = ["Hoe gaat het?", "", "The weather is nice today.", "344544"]

        for input_text in test_en_sentences:
            with self.subTest(input_text=input_text):
                input_ids = tokenizer.encode(input_text, return_tensors="pt")
                output_ids = model.generate(input_ids)
                output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

                print(f"Input: {input_text}")
                print(f"Output: {output_text}")

                detected_language = detect(input_text)
                self.assertEqual(detected_language, 'en', f"Input '{input_text}' is not English")
                self.assertTrue(len(output_text) > 0, f"Output for '{input_text}' should not be empty")
                self.assertTrue(len(input_text) > 0, f"Input: '{input_text}' should not be empty")
                self.assertIsInstance(input_text, str, f"Input: '{input_text} should be String")
                self.assertIsInstance(output_text, str, f"Output: '{output_text} should be String")
                self.assertTrue(len(input_ids[0]) <= 128, f"Input for '{input_ids}' should not be longer than 128 tokens")


unittest.main(argv=[''], exit=False)


C:\Users\matti\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
FF

Input: Hoe gaat het?
Output: Hoe gaat het?


E

Input: 
Output: Het is echt.


E

Input: The weather is nice today.
Output: Het weer is leuk vandaag.
Input: 344544
Output: 344544


E
ERROR: test_model_prediction (__main__.TestMachineTranslation.test_model_prediction) (input_text='')
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\matti\AppData\Local\Temp\ipykernel_36356\926538750.py", line 60, in test_model_prediction
    detected_language = detect(input_text)
                        ^^^^^^^^^^^^^^^^^^
  File "C:\Users\matti\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langdetect\detector_factory.py", line 130, in detect
    return detector.detect()
           ^^^^^^^^^^^^^^^^^
  File "C:\Users\matti\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langdetect\detector.py", line 136, in detect
    probabilities = self.get_probabilities()
                    ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\matti\AppData\Local\Packages\PythonSoftwa

In [ ]:
sentence = "Hi, I’m Winston, a cuddly rabbit that was once the proud assistant of a forgetful magician named Merlin. One day, Merlin put me in his hat and forgot. I found myself in a land of mismatched socks, lost keys, and odd handkerchiefs. With some effort (and nibbling on some old capes) , I finally made my way out and hopped over to the shelter, hoping for a quieter life away from magical blunders. Now, I’m looking for a home where the only tricks are those involving carrot treats."

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ------------------------------------- 981.5/981.5 kB 15.3 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993251 sha256=dfba8a25de6ea20a02e16a3e2db6e073ee18a1e01ae66e28ca5b86ca1ca69a02
  Stored in directory: c:\users\matti\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local\pip\cache\wheels\0a\f2\b2\e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [ ]:
class TestMachineTranslation(unittest.TestCase):
    def setUp(self):
        self.tokenizer = MarianTokenizer.from_pretrained("./marian_en_nl_finetuned_10000")
        self.model = MarianMTModel.from_pretrained("./marian_en_nl_finetuned_10000")

    def test_translation_time(self):
        sentences = [
                "This is a sentence to translate from English to Dutch.",
                "Hi, I’m Winston, a cuddly rabbit that was once the proud assistant of a forgetful magician named Merlin. One day, Merlin put me in his hat and forgot. I found myself in a land of mismatched socks, lost keys, and odd handkerchiefs. With some effort (and nibbling on some old capes) , I finally made my way out and hopped over to the shelter, hoping for a quieter life away from magical blunders. Now, I’m looking for a home where the only tricks are those involving carrot treats.",
                "It was easy to spot her. All you needed to do was look at her socks. They were never a matching pair. One would be green while the other would be blue. One would reach her knee while the other barely touched her ankle. Every other part of her was perfect, but never the socks. They were her micro act of rebellion. You can decide what you want to do in life, but I suggest doing something that creates. Something that leaves a tangible thing once you're done. That way even after you're gone, you will still live on in the things you created. According to the caption on the bronze marker placed by the Multnomah Chapter of the Daughters of the American Revolution on May 12, 1939, “College Hall (is) the oldest building in continuous use for Educational purposes west of the Rocky Mountains. Here were educated men and women who have won recognition throughout the world in all the learned professions.” The leather jacked showed the scars of being his favorite for years. It wore those scars with pride, feeling that they enhanced his presence rather than diminishing it. The scars gave it character and had not overwhelmed to the point that it had become ratty. The jacket was in its prime and it knew it. It was the best compliment that he'd ever received although the person who gave it likely never knew. It had been an off-hand observation on his ability to hold a conversation and actually add pertinent information to it on practically any topic. Although he hadn't consciously strived to be able to do so, he'd started to voraciously read the news when he couldn't keep up on topics his friends discussed because their conversations went above his head. The fact that someone had noticed enough to compliment him that he could talk intelligently about many topics meant that he had succeeded in his quest to be better informed. There wasn't a whole lot more that could be done. It had become a wait-and-see situation with the final results no longer in her control. That didn't stop her from trying to control the situation. She demanded that things be done as she desperately tried to control what couldn't be. Don't forget that gifts often come with costs that go beyond their purchase price. When you purchase a child the latest smartphone, you're also committing to a monthly phone bill. When you purchase the latest gaming system, you're likely not going to be satisfied with the games that come with it for long and want to purchase new titles to play. When you buy gifts it's important to remember that some come with additional costs down the road that can be much more expensive than the initial gift itself. He took a sip of the drink. He wasn't sure whether he liked it or not, but at this moment it didn't matter. She had made it especially for him so he would have forced it down even if he had absolutely hated it. That's simply the way things worked. She made him a new-fangled drink each day and he took a sip of it and smiled, saying it was excellent. There was nothing to indicate Nancy was going to change the world. She looked like an average girl going to an average high school. It was the fact that everything about her seemed average that would end up becoming her superpower. I'm so confused by your ridiculous meltdown that I must insist on some sort of explanation for your behavior towards me. It just doesn't make any sense. There's no way that I deserved the treatment you gave me without an explanation or an apology for how out of line you have been. I'm heading back to Colorado tomorrow after being down in Santa Barbara over the weekend for the festival there.  ",
                ]
        for input_text in sentences:
            with self.subTest(input_text=input_text):
                print(f"Sentence input: {input_text}")
                start = time.time()
                translated_sentence = translate(input_text, model, tokenizer)
                elapsed_time = time.time() - start
                print(f"Sentence output: {translated_sentence}, time elapsed: {elapsed_time}")
                self.assertLess(elapsed_time, 5, "Translation took more than 5 seconds")

unittest.main(argv=[''], exit=False)

C:\Users\matti\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Sentence input: This is a sentence to translate from English to Dutch.
Sentence output: Dit is een zin om van Engels naar Dutch te vertellen., time elapsed: 0.6620144844055176
Sentence input: Hi, I’m Winston, a cuddly rabbit that was once the proud assistant of a forgetful magician named Merlin. One day, Merlin put me in his hat and forgot. I found myself in a land of mismatched socks, lost keys, and odd handkerchiefs. With some effort (and nibbling on some old capes) , I finally made my way out and hopped over to the shelter, hoping for a quieter life away from magical blunders. Now, I’m looking for a home where the only tricks are those involving carrot treats.
Sentence output: Hoi, ik ben Winston, een knullig konijn dat was eens de trotse assistant van een vergeten tovenaar, Merlin. Op een dag, Merlin putte me in zijn muts en was vergeten. Ik vonden mezelf in een land van ongelijke sokken, verloren keigen en oneven handchiefs. Met een enkele plaats (en knabbelen aan een aantal oude 

F
FAIL: test_translation_time (__main__.TestMachineTranslation.test_translation_time) (input_text="It was easy to spot her. All you needed to do was look at her socks. They were never a matching pair. One would be green while the other would be blue. One would reach her knee while the other barely touched her ankle. Every other part of her was perfect, but never the socks. They were her micro act of rebellion. You can decide what you want to do in life, but I suggest doing something that creates. Something that leaves a tangible thing once you're done. That way even after you're gone, you will still live on in the things you created. According to the caption on the bronze marker placed by the Multnomah Chapter of the Daughters of the American Revolution on May 12, 1939, “College Hall (is) the oldest building in continuous use for Educational purposes west of the Rocky Mountains. Here were educated men and women who have won recognition throughout the world in all the learned profession

Sentence output: Het was gemakkelijk om haar te spotten. Alles wat je moet doen was kijk naar haar sokken. Ze waren nooit een matching paar. De ene zou green zijn, terwijl de andere bijn haar knies zou kunnen bereiken, terwijl de andere haar zelfs zelfs zelfs haar zelfs zelfs zelfs haar zelfs haar zelfs zelfs haar zoogje aan tot... elk andere deelde deel was perfect, maar nooit de sokken. Ze zouden haar micro-actie van rebellie zijn. Je kunt beslisken wat je wilt doen in het leven, maar ik voors het het het schiep heeft geschappen., time elapsed: 5.315623044967651
